In [ ]:
!pip -q install pandas numpy openpyxl requests reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.4 MB/s eta 0:00:00


In [2]:
# ============================================================
# Yuvaintern - Week 1
# Data Gathering, Cleaning, and Preprocessing
# Student: Aashifa
# Dataset: UCI Online Retail
# ============================================================

import os
import json
import zipfile
import requests
import numpy as np
import pandas as pd

from io import BytesIO
from IPython.display import display
from google.colab import files

# ------------------------------------------------------------
# 1. CREATE PROJECT FOLDERS
# ------------------------------------------------------------

os.makedirs("data", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)
os.makedirs("reports", exist_ok=True)

# ------------------------------------------------------------
# 2. DOWNLOAD PUBLIC DATASET
# ------------------------------------------------------------

url = (
    "https://archive.ics.uci.edu/"
    "ml/machine-learning-databases/00352/Online%20Retail.xlsx"
)

print("Downloading UCI Online Retail dataset...")

response = requests.get(url, timeout=180)
response.raise_for_status()

raw = pd.read_excel(BytesIO(response.content))

df = raw.copy()

original_rows = len(df)
original_columns = len(df.columns)

print("\nOriginal dataset shape:", raw.shape)
print("\nFirst five records:")
display(raw.head())

# ------------------------------------------------------------
# 3. INITIAL DATA EXPLORATION
# ------------------------------------------------------------

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types before cleaning:")
print(df.dtypes)

print("\nMissing values before cleaning:")
print(df.isnull().sum())

duplicates_before = int(df.duplicated().sum())

print("\nDuplicate rows before cleaning:", duplicates_before)

print("\nNumerical summary:")
display(df.describe(include="all").T)

missing_before = {
    str(column): int(count)
    for column, count in df.isna().sum().items()
}

# ------------------------------------------------------------
# 4. REMOVE EXACT DUPLICATES
# ------------------------------------------------------------

df = df.drop_duplicates().copy()

print("\nShape after duplicate removal:", df.shape)

# ------------------------------------------------------------
# 5. CLEAN TEXT AND IDENTIFIER COLUMNS
# ------------------------------------------------------------

text_columns = ["InvoiceNo", "StockCode", "Description", "Country"]

for column in text_columns:
    df[column] = df[column].map(
        lambda value: value.strip()
        if isinstance(value, str)
        else value
    )

# Keep identifiers as text rather than numerical measurements.
df["InvoiceNo"] = df["InvoiceNo"].astype("string")
df["StockCode"] = df["StockCode"].astype("string")
df["Description"] = df["Description"].astype("string")
df["Country"] = df["Country"].astype("string")

# ------------------------------------------------------------
# 6. CONVERT DATA TYPES
# ------------------------------------------------------------

df["Quantity"] = pd.to_numeric(
    df["Quantity"], errors="coerce"
)

df["UnitPrice"] = pd.to_numeric(
    df["UnitPrice"], errors="coerce"
)

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

# CustomerID is an identifier, not a numerical measurement.
customer_numeric = pd.to_numeric(
    df["CustomerID"], errors="coerce"
)

df["CustomerID"] = (
    customer_numeric
    .round()
    .astype("Int64")
    .astype("string")
)

# ------------------------------------------------------------
# 7. HANDLE MISSING VALUES
# ------------------------------------------------------------

essential_columns = [
    "InvoiceNo",
    "StockCode",
    "InvoiceDate",
    "Quantity",
    "UnitPrice",
    "Country"
]

missing_essential_rows = int(
    df[essential_columns].isna().any(axis=1).sum()
)

print(
    "\nRows with missing essential fields:",
    missing_essential_rows
)

# Remove records missing essential transaction information.
df = df.dropna(subset=essential_columns).copy()

# Replace missing or blank product descriptions.
df["Description"] = (
    df["Description"]
    .fillna("Unknown product")
    .str.strip()
)

df.loc[
    df["Description"].eq(""),
    "Description"
] = "Unknown product"

# Missing CustomerID values are retained.
# Customer IDs are not invented or imputed.

# ------------------------------------------------------------
# 8. HANDLE INVALID NUMERICAL VALUES
# ------------------------------------------------------------

zero_quantity_removed = int(
    (df["Quantity"] == 0).sum()
)

negative_price_removed = int(
    (df["UnitPrice"] < 0).sum()
)

print("\nZero-quantity records:", zero_quantity_removed)
print("Negative-price records:", negative_price_removed)

# Zero quantity records provide no quantity sold.
# Negative prices are treated as invalid for this project.
df = df[
    (df["Quantity"] != 0) &
    (df["UnitPrice"] >= 0)
].copy()

# Retain negative quantities because they can represent returns.
# Zero prices are retained for review rather than assumed invalid.

# ------------------------------------------------------------
# 9. IDENTIFY CANCELLATIONS AND RETURNS
# ------------------------------------------------------------

df["IsCancellation"] = (
    df["InvoiceNo"]
    .str.upper()
    .str.startswith("C", na=False)
)

df["IsReturn"] = df["Quantity"] < 0

# ------------------------------------------------------------
# 10. FEATURE ENGINEERING
# ------------------------------------------------------------

df["LineTotal"] = (
    df["Quantity"] * df["UnitPrice"]
)

# ------------------------------------------------------------
# 11. DETECT OUTLIERS USING THE IQR METHOD
# ------------------------------------------------------------

def calculate_iqr_bounds(series):
    series = series.dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    return float(lower_bound), float(upper_bound)


# Evaluate positive quantities and positive prices separately.
positive_quantities = df.loc[
    df["Quantity"] > 0, "Quantity"
]

positive_prices = df.loc[
    df["UnitPrice"] > 0, "UnitPrice"
]

quantity_lower, quantity_upper = calculate_iqr_bounds(
    positive_quantities
)

price_lower, price_upper = calculate_iqr_bounds(
    positive_prices
)

df["QuantityOutlier"] = (
    (df["Quantity"] > 0) &
    (
        (df["Quantity"] < quantity_lower) |
        (df["Quantity"] > quantity_upper)
    )
)

df["PriceOutlier"] = (
    (df["UnitPrice"] > 0) &
    (
        (df["UnitPrice"] < price_lower) |
        (df["UnitPrice"] > price_upper)
    )
)

quantity_outliers = int(df["QuantityOutlier"].sum())
price_outliers = int(df["PriceOutlier"].sum())

print("\nQuantity IQR bounds:")
print(quantity_lower, "to", quantity_upper)

print("\nUnitPrice IQR bounds:")
print(price_lower, "to", price_upper)

print("\nQuantity outliers flagged:", quantity_outliers)
print("Price outliers flagged:", price_outliers)

# Outliers are flagged, not automatically deleted.
# Large purchases may be genuine business transactions.

# ------------------------------------------------------------
# 12. SORT AND SAVE THE CLEANED DATASET
# ------------------------------------------------------------

df = df.sort_values(
    ["InvoiceDate", "InvoiceNo", "StockCode"],
    na_position="last"
).reset_index(drop=True)

csv_path = "data/cleaned_online_retail.csv"

df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 13. VALIDATE THE CLEANED DATASET
# ------------------------------------------------------------

missing_after = {
    str(column): int(count)
    for column, count in df.isna().sum().items()
}

duplicates_after = int(df.duplicated().sum())

print("\nFinal dataset shape:", df.shape)
print("\nData types after cleaning:")
print(df.dtypes)

print("\nMissing values after cleaning:")
print(df.isna().sum())

print("\nDuplicate rows after cleaning:", duplicates_after)

print("\nCleaned data preview:")
display(df.head())

# ------------------------------------------------------------
# 14. SAVE ACTUAL CLEANING RESULTS
# ------------------------------------------------------------

results = {
    "project": "Yuvaintern Week 1",
    "student": "Aashifa",
    "dataset": "UCI Online Retail",
    "source": (
        "https://archive.ics.uci.edu/dataset/352/online+retail"
    ),
    "original_rows": int(original_rows),
    "original_columns": int(original_columns),
    "duplicates_before": int(duplicates_before),
    "missing_before": missing_before,
    "missing_essential_rows_removed": int(
        missing_essential_rows
    ),
    "zero_quantity_records_removed": int(
        zero_quantity_removed
    ),
    "negative_price_records_removed": int(
        negative_price_removed
    ),
    "final_rows": int(len(df)),
    "final_columns": int(len(df.columns)),
    "duplicates_after": int(duplicates_after),
    "missing_after": missing_after,
    "quantity_outliers_flagged": int(quantity_outliers),
    "price_outliers_flagged": int(price_outliers),
    "missing_customer_ids_retained": int(
        df["CustomerID"].isna().sum()
    ),
    "cancellations_retained": int(
        df["IsCancellation"].sum()
    ),
    "returns_retained": int(
        df["IsReturn"].sum()
    ),
    "date_start": str(df["InvoiceDate"].min()),
    "date_end": str(df["InvoiceDate"].max()),
    "quantity_iqr_bounds": [
        quantity_lower, quantity_upper
    ],
    "price_iqr_bounds": [
        price_lower, price_upper
    ],
    "cleaning_notes": [
        "Exact duplicate records removed",
        "Text whitespace trimmed",
        "Data types converted",
        "Missing essential fields removed",
        "Missing descriptions replaced",
        "Missing customer IDs retained",
        "Zero quantities removed",
        "Negative unit prices removed",
        "Negative quantities retained",
        "Potential outliers flagged, not deleted",
        "LineTotal calculated as Quantity times UnitPrice"
    ]
}

json_path = "reports/cleaning_results.json"

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(results, file, indent=4, ensure_ascii=False)

# ------------------------------------------------------------
# 15. CREATE ZIP FOR PROJECT FILES
# ------------------------------------------------------------

zip_path = "Week1-Data-Cleaning-Preprocessing.zip"

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as archive:

    archive.write(
        csv_path,
        arcname="data/cleaned_online_retail.csv"
    )

    archive.write(
        json_path,
        arcname="reports/cleaning_results.json"
    )

# ------------------------------------------------------------
# 16. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("YUVaintern WEEK 1 PROJECT COMPLETED")
print("=" * 55)

print("Student: Aashifa")
print("Original rows:", original_rows)
print("Final rows:", len(df))
print("Original columns:", original_columns)
print("Final columns:", len(df.columns))
print("Rows removed:", original_rows - len(df))
print("Duplicates remaining:", duplicates_after)
print("CSV file:", csv_path)
print("Results file:", json_path)
print("ZIP file:", zip_path)

print("\nDownload the project ZIP:")

files.download(zip_path)


Original dataset shape: (541909, 8)

First five records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom



Column names:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data types before cleaning:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Missing values before cleaning:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicate rows before cleaning: 5268

Numerical summary:


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
InvoiceNo,541909.0,25900.0,573585.0,1114.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,541909.0,NaN,NaN,NaN,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,NaN,NaN,NaN,2011-07-04 13:34:57.156386048,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,NaN,NaN,NaN,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,406829.0,NaN,NaN,NaN,15287.69057,12346.0,13953.0,15152.0,16791.0,18287.0,1713.600303
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Shape after duplicate removal: (536641, 8)

Rows with missing essential fields: 0

Zero-quantity records: 0
Negative-price records: 2

Quantity IQR bounds:
-14.0 to 26.0

UnitPrice IQR bounds:
-3.0700000000000003 to 8.45

Quantity outliers flagged: 27355
Price outliers flagged: 39448

Final dataset shape: (536639, 13)

Data types after cleaning:
InvoiceNo          string[python]
StockCode          string[python]
Description        string[python]
Quantity                    int64
InvoiceDate        datetime64[ns]
UnitPrice                 float64
CustomerID         string[python]
Country            string[python]
IsCancellation            boolean
IsReturn                     bool
LineTotal                 float64
QuantityOutlier              bool
PriceOutlier                 bool
dtype: object

Missing values after cleaning:
InvoiceNo               0
StockCode               0
Description             0
Quantity                0
InvoiceDate             0
UnitPrice               0
Custome

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancellation,IsReturn,LineTotal,QuantityOutlier,PriceOutlier
0,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,False,False,25.50,False,False
1,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,False,False,15.30,False,False
2,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,20.34,False,False
3,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,20.34,False,False
4,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,20.34,False,False



YUVaintern WEEK 1 PROJECT COMPLETED
Student: Aashifa
Original rows: 541909
Final rows: 536639
Original columns: 8
Final columns: 13
Rows removed: 5270
Duplicates remaining: 0
CSV file: data/cleaned_online_retail.csv
Results file: reports/cleaning_results.json
ZIP file: Week1-Data-Cleaning-Preprocessing.zip

Download the project ZIP:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>